# Combined Feature Engineering

In [1]:
import pandas as pd
import numpy as np
import os, sys, json
import warnings
warnings.filterwarnings('ignore')

sys.path.append("..")

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import joblib

PROCESSED_DIR = "../data/processed"
MASTER_PATH = os.path.join(PROCESSED_DIR, "labeled_master_combined.csv")
RANDOM_STATE = 42

In [2]:
df = pd.read_csv(MASTER_PATH)
print(f"Loaded combined master: {df.shape}")
print(f"Columns: {len(df.columns)}")

Loaded combined master: (5139, 633)
Columns: 633


In [3]:
# Remove non-reference duplicate lab variables that cause leakage
DUPLICATE_LAB_VARS = ["LBXSGL", "LBXSCH", "LBXSTR"]
df = df.drop(columns=[c for c in DUPLICATE_LAB_VARS if c in df.columns])
print(f"Removed duplicate lab variables. New shape: {df.shape}")

Removed duplicate lab variables. New shape: (5139, 630)


In [4]:
# ====================================================
# ADMINISTRATIVE VARIABLES (never used as predictors)
# ====================================================
ADMIN_VARS = [
    "SEQN", "WTMEC2YR", "SDMVPSU", "SDMVSTRA",
    "RIDSTATR", "BMDSTATS", "WTSAF2YR",
]

# ====================================================
# LABEL COLUMNS (created earlier)
# ====================================================
LABEL_COLS = [c for c in df.columns if c.startswith("label_")]

# ====================================================
# CURATED FEATURE LIST (from Master_Feature_Catalog.xlsx)
# ====================================================
SELECTED_FEATURES = [
    # DEMO
    "RIDAGEYR", "RIAGENDR", "RIDRETH3", "INDFMPIR", "DMDEDUC2", "DMDMARTL",
    "DMDBORN4", "DMDYRSUS", "INDHHIN2", "DMDHHSIZ", "RIDEXMON",
    # BMX
    "BMXWT", "BMXHT", "BMXBMI", "BMXWAIST", "BMXHIP", "BMXARMC", "BMXARML", "BMXLEG",
    # GHB
    "LBXGH",
    # GLU
    "LBXGLU",
    # BPX
    "BPXSY1", "BPXDI1", "BPXSY2", "BPXDI2", "BPXSY3", "BPXDI3", "BPXSY4", "BPXDI4",
    "BPXPLS", "BPXPULS",
    # BIOPRO
    "LBXSCR", "LBXSBU", "LBXSAL", "LBXSTP", "LBXSCA", "LBXSNASI", "LBXSKSI",
    "LBXSCLSI", "LBXSTB", "LBXSAPSI", "LBXSATSI", "LBXSASSI", "LBXSGTSI",
    "LBXSPH", "LBXSC3SI", "LBXSUA", "LBXSIR", "LBXSGB", "LBXSCK", "LBXSLDSI",
    "LBXSOSSI",
    # DIQ (excluding DIQ010 – pure label)
    "DIQ160", "DID040", "DIQ080",
    # KIQ
    "KIQ022", "KIQ025", "KIQ026",
    # SMQ
    "SMQ020", "SMD030", "SMQ040", "SMD650", "SMD057", "SMQ890", "SMQ900", "SMQ910",
    # ALQ
    "ALQ111", "ALQ121", "ALQ130", "ALQ142", "ALQ151",
    # PAQ
    "PAQ605", "PAQ620", "PAQ635", "PAQ650", "PAQ665", "PAD680",
    # DBQ
    "DBQ700", "DBQ197", "DBD895", "DBD900", "DBD905", "DBD910",
    # MCQ
    "MCQ160B", "MCQ160C", "MCQ160D", "MCQ160E", "MCQ160F",
    "MCQ160L", "MCQ300C", "MCQ300A",
    # CBC
    "LBXWBCSI", "LBXHGB", "LBXRDW", "LBXPLTSI", "LBXLYPCT", "LBXNEPCT",
    # Lipids
    "LBDHDD", "LBXTC", "LBXTR",
    # Other labs
    "LBXIN", "LBXHSCRP", "URDACT",
    # HSQ
    "HSD010"
]

# ====================================================
# TEMPORARY COLUMNS (needed for derived features but not predictors)
# ====================================================
TEMP_DERIVED_SOURCE_COLS = [
    "DIQ010",      # diabetes diagnosis (for ASCVD risk, not used as predictor)
    "BPQ020",      # hypertension treatment (for ASCVD risk, not used as predictor)
    "mean_sbp",    # derived earlier, needed for ASCVD and met syndrome count
    "mean_dbp",    # derived earlier, needed for met syndrome count
    "eGFR",        # derived earlier, needed for eGFR stage
]

# ====================================================
# KEEP SELECTED FEATURES + TEMP SOURCES + ADMIN + LABELS
# ====================================================
cols_to_keep = SELECTED_FEATURES + TEMP_DERIVED_SOURCE_COLS + ADMIN_VARS + LABEL_COLS
df = df[cols_to_keep].copy()
print(f"After curated feature selection, df shape: {df.shape}")

# ====================================================
# PREDICTOR COLUMNS (initially only selected features)
# ====================================================
PREDICTOR_COLS = SELECTED_FEATURES.copy()
print(f"Predictor columns (before derived): {len(PREDICTOR_COLS)}")

After curated feature selection, df shape: (5139, 123)
Predictor columns (before derived): 104


In [5]:
# NLR
df["NLR"] = df["LBXNEPCT"] / df["LBXLYPCT"].replace(0, np.nan)
df["NLR"] = df["NLR"].replace([np.inf, -np.inf], np.nan)

# HOMA-IR
df["HOMA_IR"] = (df["LBXIN"] * df["LBXGLU"]) / 405

# WHR
df["WHR"] = df["BMXWAIST"] / df["BMXHIP"].replace(0, np.nan)

DERIVED_COLS = ["NLR", "HOMA_IR", "WHR"]
PREDICTOR_COLS.extend(DERIVED_COLS)
print(f"Predictor columns (after derived): {len(PREDICTOR_COLS)}")

Predictor columns (after derived): 107


In [6]:
# ====================================================
# ADDITIONAL CLINICAL DERIVED FEATURES
# ====================================================

# 1. ASCVD 10-year risk score (simplified, using pooled cohort equations coefficients)
def ascvd_risk(row):
    """
    Simplified ASCVD risk calculation for demonstration.
    Uses age, sex, total cholesterol, HDL, systolic BP, smoking, diabetes, hypertension.
    Returns 10-year risk percentage.
    """
    age = row["RIDAGEYR"]
    sex = row["RIAGENDR"]  # 1=Male, 2=Female
    tc = row["LBXTC"]
    hdl = row["LBDHDD"]
    sbp = row["mean_sbp"]
    smoker = 1 if row.get("SMQ040", 0) in [1, 2] else 0
    diabetic = 1 if row.get("DIQ010", 2) == 1 else 0
    ht_treated = 1 if row.get("BPQ020", 2) == 1 else 0

    # We'll use a simplified linear combination (not the full Pooled Cohort equation)
    # but directionally correct. In real use, implement the full equation.
    risk = (age * 0.05) + (tc * 0.002) - (hdl * 0.003) + (sbp * 0.004) + (smoker * 0.15) + (diabetic * 0.2) + (ht_treated * 0.1)
    return max(0.0, min(risk, 1.0))  # clip to [0,1]

df["ASCVD_risk"] = df.apply(ascvd_risk, axis=1)

# 2. Metabolic syndrome component count (0-5)
def met_syndrome_count(row):
    count = 0
    if (row["RIAGENDR"] == 1 and row["BMXWAIST"] > 102) or (row["RIAGENDR"] == 2 and row["BMXWAIST"] > 88):
        count += 1
    if not pd.isna(row["LBXTR"]) and row["LBXTR"] >= 150:
        count += 1
    if not pd.isna(row["LBDHDD"]):
        hdl_low = (row["LBDHDD"] < 40) if row["RIAGENDR"] == 1 else (row["LBDHDD"] < 50)
        if hdl_low:
            count += 1
    if not pd.isna(row["mean_sbp"]) and not pd.isna(row["mean_dbp"]):
        if row["mean_sbp"] >= 130 or row["mean_dbp"] >= 85:
            count += 1
    if not pd.isna(row["LBXGLU"]) and row["LBXGLU"] >= 100:
        count += 1
    return count

df["met_syndrome_count"] = df.apply(met_syndrome_count, axis=1)

# 3. eGFR category (G-stage)
def egfr_stage(egfr):
    if pd.isna(egfr):
        return np.nan
    if egfr >= 90:
        return 1
    elif egfr >= 60:
        return 2
    elif egfr >= 45:
        return 3
    elif egfr >= 30:
        return 4
    else:
        return 5

df["eGFR_stage"] = df["eGFR"].apply(egfr_stage)

# 4. Obesity class from BMI
def obesity_class(bmi):
    if pd.isna(bmi):
        return np.nan
    if bmi < 18.5:
        return 0  # underweight
    elif bmi < 25:
        return 1  # normal
    elif bmi < 30:
        return 2  # overweight
    elif bmi < 35:
        return 3  # obese class I
    elif bmi < 40:
        return 4  # obese class II
    else:
        return 5  # obese class III

df["obesity_class"] = df["BMXBMI"].apply(obesity_class)

# 5. Smoking pack-years (approximate)
# For simplicity, use age started and current cigarettes per day.
def pack_years(row):
    if pd.isna(row.get("SMQ020", np.nan)):
        return 0
    if row["SMQ020"] != 1:
        return 0
    start_age = row.get("SMD030", np.nan)
    current_age = row["RIDAGEYR"]
    if pd.isna(start_age) or start_age <= 0:
        return 0
    years_smoked = max(0, current_age - start_age)
    cigs_per_day = row.get("SMD650", 0)
    if pd.isna(cigs_per_day):
        cigs_per_day = 0
    return (cigs_per_day * years_smoked) / 20  # pack-years

df["pack_years"] = df.apply(pack_years, axis=1)

# Add new derived features to SELECTED_FEATURES and PREDICTOR_COLS
NEW_DERIVED = ["ASCVD_risk", "met_syndrome_count", "eGFR_stage", "obesity_class", "pack_years"]
SELECTED_FEATURES.extend(NEW_DERIVED)
PREDICTOR_COLS.extend(NEW_DERIVED)

print(f"Total predictors after additional derived features: {len(PREDICTOR_COLS)}")

Total predictors after additional derived features: 112


In [7]:
LEAKAGE_MAP = {
    "label_diabetes": [
        "LBXGH", "LBXGLU", "DIQ010", "HOMA_IR",
        "LBXSGL",               # duplicate serum glucose
        "met_syndrome_count",   # uses glucose component
        "ASCVD_risk",           # uses diabetes diagnosis (DIQ010)
        "DIQ160", "DID040", "DIQ080"   # diabetes diagnosis/treatment
    ],
    "label_hypertension": [
        "BPXSY1", "BPXSY2", "BPXSY3", "BPXSY4",
        "BPXDI1", "BPXDI2", "BPXDI3", "BPXDI4",
        "BPXPLS", "BPXPULS", "BPQ020",
        "mean_sbp", "mean_dbp",
        "ASCVD_risk"            # uses systolic BP
    ],
    "label_ckd": [
        "LBXSCR", "eGFR", "KIQ022", "KIQ025", "URDACT",
        "eGFR_stage"            # derived from eGFR
    ],
    "label_dyslipidemia": [
        "LBXTC", "LBDHDD", "LBXTR",
        "LBXSCH", "LBXSTR",     # duplicate lipid variables
        "ASCVD_risk"            # uses total cholesterol and HDL
    ],
    "label_cvd": [
        "MCQ160B", "MCQ160C", "MCQ160D", "MCQ160E", "MCQ160F",
        "MCQ300A", "MCQ300B", "MCQ300C"   # age at diagnosis for events
    ],
    "label_metabolic_syndrome": [
        "BMXWAIST", "LBXTR", "LBDHDD",
        "BPXSY1", "BPXSY2", "BPXSY3", "BPXSY4",
        "BPXDI1", "BPXDI2", "BPXDI3", "BPXDI4",
        "mean_sbp", "mean_dbp",
        "LBXGLU", "WHR",
        "LBXSGL", "LBXSTR",
        "met_syndrome_count",   # direct count of label components
        "ASCVD_risk"            # uses HDL and blood pressure
    ],
    "label_liver_disease": [
        "LBXSATSI", "LBXSASSI", "LBXSGTSI", "BMXBMI",
        "obesity_class"         # derived from BMI
    ],
}

In [8]:
feature_sets = {}
label_sets = {}

for label_col in LABEL_COLS:
    mask = df[label_col].notna()
    y = df.loc[mask, label_col].astype(int)
    
    exclude = LEAKAGE_MAP.get(label_col, [])
    predictor_cols = [c for c in PREDICTOR_COLS if c not in exclude]
    
    X = df.loc[mask, predictor_cols].copy()
    
    # Convert all to numeric (coerce errors)
    for col in X.columns:
        X[col] = pd.to_numeric(X[col], errors='coerce')
    
    # Drop columns that became all NaN
    X = X.dropna(axis=1, how='all')
    
    feature_sets[label_col] = X
    label_sets[label_col] = y
    print(f"{label_col}: X.shape={X.shape}, y positives={y.sum()}, negatives={len(y)-y.sum()}")

label_diabetes: X.shape=(5082, 104), y positives=1133, negatives=3949
label_hypertension: X.shape=(5139, 101), y positives=2893, negatives=2246
label_ckd: X.shape=(5139, 107), y positives=473, negatives=4666
label_dyslipidemia: X.shape=(4837, 108), y positives=2844, negatives=1993
label_cvd: X.shape=(5139, 105), y positives=633, negatives=4506
label_metabolic_syndrome: X.shape=(4315, 97), y positives=1613, negatives=2702
label_liver_disease: X.shape=(5126, 107), y positives=730, negatives=4396


In [9]:
split_data = {}
feature_names = {}

for label_col in LABEL_COLS:
    X = feature_sets[label_col]
    y = label_sets[label_col]
    
    # Impute missing with median
    imputer = SimpleImputer(strategy='median')
    X_imputed = imputer.fit_transform(X)
    
    # Split BEFORE scaling to avoid leakage
    X_train_raw, X_test_raw, y_train, y_test = train_test_split(
        X_imputed, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
    )
    
    # Scale using training set only
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_raw)
    X_test = scaler.transform(X_test_raw)
    
    disease_name = label_col.replace("label_", "")
    split_data[disease_name] = (X_train, X_test, y_train, y_test)
    feature_names[disease_name] = X.columns.tolist()
    
    # Save arrays (overwrite old)
    np.save(os.path.join(PROCESSED_DIR, f"{disease_name}_X_train.npy"), X_train)
    np.save(os.path.join(PROCESSED_DIR, f"{disease_name}_X_test.npy"), X_test)
    np.save(os.path.join(PROCESSED_DIR, f"{disease_name}_y_train.npy"), y_train)
    np.save(os.path.join(PROCESSED_DIR, f"{disease_name}_y_test.npy"), y_test)
    
    # Save scaler
    scaler_path = os.path.join(PROCESSED_DIR, f"{disease_name}_scaler.pkl")
    joblib.dump(scaler, scaler_path)
    
    print(f"{disease_name}: train={X_train.shape}, test={X_test.shape}, pos_train={y_train.sum()}, pos_test={y_test.sum()}")

# Save feature names (mapping label_disease -> list)
feature_names_json = {}
for disease in feature_names:
    feature_names_json[f"label_{disease}"] = feature_names[disease]

with open(os.path.join(PROCESSED_DIR, "feature_names.json"), "w") as f:
    json.dump(feature_names_json, f, indent=2)

print("\nAll processed data and scalers saved.")

diabetes: train=(4065, 104), test=(1017, 104), pos_train=906, pos_test=227
hypertension: train=(4111, 101), test=(1028, 101), pos_train=2314, pos_test=579
ckd: train=(4111, 107), test=(1028, 107), pos_train=378, pos_test=95
dyslipidemia: train=(3869, 108), test=(968, 108), pos_train=2275, pos_test=569
cvd: train=(4111, 105), test=(1028, 105), pos_train=506, pos_test=127
metabolic_syndrome: train=(3452, 97), test=(863, 97), pos_train=1290, pos_test=323
liver_disease: train=(4100, 107), test=(1026, 107), pos_train=584, pos_test=146

All processed data and scalers saved.


In [10]:
import json
import numpy as np
import os

PROCESSED_DIR = "../data/processed"

# 1. Check feature_names.json vs leakage_map
with open(os.path.join(PROCESSED_DIR, "feature_names.json")) as f:
    feature_names = json.load(f)
with open(os.path.join(PROCESSED_DIR, "leakage_map.json")) as f:
    leakage_map = json.load(f)

print("=== Leakage Check ===")
for label, feats in feature_names.items():
    excluded = set(leakage_map.get(label, []))
    leaked = [f for f in feats if f in excluded]
    if leaked:
        print(f"❌ {label}: LEAKAGE FOUND: {leaked}")
    else:
        print(f"✅ {label}: clean")

# 2. Check feature count consistency
print("\n=== Feature Count Check ===")
for disease in ["diabetes","hypertension","ckd","dyslipidemia","cvd","metabolic_syndrome","liver_disease"]:
    X_train = np.load(f"{PROCESSED_DIR}/{disease}_X_train.npy")
    feats = feature_names[f"label_{disease}"]
    if X_train.shape[1] == len(feats):
        print(f"✅ {disease}: {X_train.shape[1]} features match")
    else:
        print(f"❌ {disease}: X has {X_train.shape[1]} columns but feature_names has {len(feats)}")

# 3. Quick correlation check for extreme leakage
from scipy.stats import pointbiserialr
print("\n=== High Correlation Check (>0.95) ===")
for disease in ["diabetes","hypertension","ckd","dyslipidemia","cvd","metabolic_syndrome","liver_disease"]:
    X_train = np.load(f"{PROCESSED_DIR}/{disease}_X_train.npy")
    y_train = np.load(f"{PROCESSED_DIR}/{disease}_y_train.npy")
    feats = feature_names[f"label_{disease}"][:X_train.shape[1]]
    for i, col in enumerate(feats):
        corr, _ = pointbiserialr(y_train, X_train[:, i])
        if abs(corr) > 0.95:
            print(f"🚨 {disease}: {col} correlation={corr:.3f}")

print("\nPre-tuning verification complete.")

=== Leakage Check ===
✅ label_diabetes: clean
✅ label_hypertension: clean
✅ label_ckd: clean
✅ label_dyslipidemia: clean
✅ label_cvd: clean
✅ label_metabolic_syndrome: clean
✅ label_liver_disease: clean

=== Feature Count Check ===
✅ diabetes: 104 features match
✅ hypertension: 101 features match
✅ ckd: 107 features match
✅ dyslipidemia: 108 features match
✅ cvd: 105 features match
✅ metabolic_syndrome: 97 features match
✅ liver_disease: 107 features match

=== High Correlation Check (>0.95) ===

Pre-tuning verification complete.
